# Capability 20: Analytical comparisons across KPIs, entities, periods, and business domains

8/8 cases passed against a real, live LLM (gateway-configured model, see `.env`). Every code cell below is real, executable code -- the same `ask()` pattern as `notebooks/demo.ipynb` -- not a mockup; the attached output is what actually happened when this ran, captured via `scripts/run_live_capability_tests.py --capability 20`. Re-running this notebook (Restart Kernel & Run All) with a live key will make new real calls.

See `tests/live/cases/cap20_analytical_comparisons.py` for these case definitions with their automated pass/fail checks, and `tests/live/live_capabilities_suite.py` for how they run as unittest assertions.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 01: compare one KPI across two zones

**✅ PASS**

In [ ]:
_ = ask('Compare revenue between North America and EMEA in 2025', '01')

01
USER: Compare revenue between North America and EMEA in 2025

[intent=comparison | sub_agents=['structured']]

AGENT: ## Revenue comparison — North America vs. EMEA (2025)

| Zone | Revenue (USD m, 2025) |
| --- | --- |
| North America | 14,208.0 |
| EMEA | 9,502.0 |

**Key takeaways**

- **North America generated more revenue than EMEA in 2025**: 14,208 m USD vs. 9,502 m USD — a difference of **4,706 m USD** in North America's favor.

**Data notes / caveats**

- These figures are the summed quarterly-grain revenue records for 2025 for the two requested zones only; other zones (e.g., Latin America, Asia Pacific) were not part of the query, so this is not a company-wide revenue total.
- The evidence contains only annual zone totals for 2025 — no quarterly split and no prior-year comparison — so I can't tell whether this gap was consistent across the year or how it changed versus 2024.

**Suggested follow-ups**

- View the **quarterly 2025 breakdown** for each zone to see when the gap

## 02: compare one zone across two periods

**✅ PASS**

In [ ]:
_ = ask("How did North America's revenue in 2025 compare to 2024?", '02')

02
USER: How did North America's revenue in 2025 compare to 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's revenue declined in 2025 compared to 2024. Based on the quarterly KPI data:

| Metric | Value |
|---|---|
| Revenue 2025 | US$14,208M |
| Revenue 2024 | US$14,655M |
| Change | –US$447M |
| Change (%) | –3.05% |

So 2025 was a down year for North America, with revenue falling **US$447M, or –3.05%, year-over-year**.

Would you like me to break down the quarterly trend within 2025 to see when the decline occurred, or compare North America's performance to other zones?

## 03: compare two KPIs for one zone

**✅ PASS**

In [ ]:
_ = ask('Compare EBITDA margin and organic revenue growth for North America in 2025', '03')

03
USER: Compare EBITDA margin and organic revenue growth for North America in 2025

[intent=data_query | sub_agents=['structured']]

AGENT: ## North America 2025: EBITDA Margin vs Organic Revenue Growth

| Quarter | EBITDA Margin | Organic Revenue Growth |
| --- | --- | --- |
| Q1 2025 | 32.3% | -4.7% |
| Q2 2025 | 35.7% | +2.2% |
| Q3 2025 | 35.1% | -0.7% |
| Q4 2025 | 28.0% | -1.0% |

**What the two metrics say together**

- **Margins were strong for most of the year, but growth was not.** EBITDA margin held in the 32–36% range through Q1–Q3 before falling to 28.0% in Q4. Organic revenue growth, by contrast, was negative in three of four quarters, with Q1 the clear low point at -4.7% and Q2 the only positive quarter at +2.2%.
- **The two series decouple in Q3.** Margin stayed near its peak (35.1%, essentially flat vs. Q2's 35.7%) while organic growth slipped back to -0.7%. That pattern suggests margin was being supported by something other than top-line expansion (e.g., cost or mix 

## 04: compare two KPIs across two zones at once

**✅ PASS**

In [ ]:
_ = ask('Compare EBITDA margin and organic revenue growth for North America versus EMEA in 2025', '04')

04
USER: Compare EBITDA margin and organic revenue growth for North America versus EMEA in 2025

[intent=comparison | sub_agents=['structured']]

AGENT: ## North America vs EMEA — 2025 (quarterly average)

| Zone | Avg EBITDA margin (%) | Avg organic revenue growth (%) |
| --- | --- | --- |
| North America | 32.775 | **−1.05** |
| EMEA | 32.55 | **3.30** |

**What stands out**

- **EBITDA margin is essentially at parity.** North America's 32.775% sits only marginally above EMEA's 32.55% (a gap of ~0.23 percentage points, computed from the table) — not a meaningful profitability difference between the two zones.
- **Organic revenue growth is where they diverge.** EMEA posted +3.30%, while North America contracted at −1.05% — a spread of ~4.35 percentage points (again, computed from the table). This is the real story: North America is holding its margin while its top line shrank, whereas EMEA is growing while defending a similar margin level.
- Practical read: EMEA shows the healthier gr

## 05: compare three zones on one KPI

**✅ PASS**

In [ ]:
_ = ask('Compare revenue across North America, EMEA, and Asia Pacific in 2025', '05')

05
USER: Compare revenue across North America, EMEA, and Asia Pacific in 2025

[intent=comparison | sub_agents=['structured']]

AGENT: ## Revenue by Zone — Full-Year 2025

| Zone | Revenue (USD million) |
| --- | --- |
| North America | $14,208 |
| EMEA | $9,502 |
| Asia Pacific | $5,694 |

**Key takeaways**

- **North America is the largest of the three zones** at $14,208m.
- **EMEA ranks second** at $9,502m.
- **Asia Pacific is the smallest** at $5,694m.

**Caveats on the data:** Figures are the SQL aggregate `SUM(revenue_usd_m)` filtered to `year = 2025` and `grain = 'quarterly'`, grouped by zone — i.e., full-year 2025 totals built from quarterly records. Only these three zones are included; other zones and non-quarterly rows are excluded. No internal documents or web evidence were retrieved, so this comparison rests on the structured KPI data alone.

*Follow-up ideas:* Would you like the same three zones broken out **by quarter** to see the intra-year trend, or a comparison against

## 06: compare a brand's document-level context across two countries

**✅ PASS**

In [ ]:
_ = ask("How does Corona's positioning differ between Mexico and other markets?", '06')

06
USER: How does Corona's positioning differ between Mexico and other markets?

[intent=comparison | sub_agents=['unstructured']]
[citations: ['DOC-011', 'DOC-012', 'DOC-004', 'DOC-008', 'DOC-014']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **Middle Americas** zone (which includes Mexico) instead, plus any Mexico-specific commentary found in the documents.

AGENT: ## Short answer

Based on the retrieved evidence, the clearest documented difference is one of **brand scope and ownership**, not consumer positioning language: AB InBev's "Corona" megabrand is defined *excluding* Mexico, because a permanent license to the Corona/Modelo brand sits with Constellation Brands in that market [DOC-012]. Outside Mexico, Corona is counted and reported as one of AB InBev's four global "megabrands" (alongside Budweiser, Stella Artois and Michelob Ultra) and is therefore positioned as a **global, multi-country growth brand*

## 07: compare quarter-over-quarter within a single zone

**✅ PASS**

In [ ]:
_ = ask('Compare Middle Americas revenue in Q4 2025 versus Q3 2025', '07')

07
USER: Compare Middle Americas revenue in Q4 2025 versus Q3 2025

[intent=comparison | sub_agents=['structured']]

AGENT: **Middle Americas revenue: Q4 2025 vs Q3 2025**

| Metric | Q3 2025 | Q4 2025 | QoQ change |
| --- | --- | --- | --- |
| Revenue (USD m) | 4,325.0 | 4,927.0 | +602.0 |
| QoQ change (%) | — | — | +13.92% |

Q4 2025 revenue for Middle Americas was **$4,927.0m**, an increase of **$602.0m** over the **$4,325.0m** recorded in Q3 2025, equivalent to sequential growth of **+13.92%**.

Scope note: these figures come directly from the structured KPI data (zone = Middle Americas, year = 2025, quarters 3 and 4), so the comparison is on the same KPI definition (`revenue_usd_m`). No document, web, or code-execution evidence was provided, so there is no qualitative commentary (e.g. price/mix or volume drivers) available to explain the increase.

Possible next steps:
- Compare Q4 2025 with Q4 2024 for a year-over-year view of Middle Americas revenue.
- Look at another KPI for th

## 08: compare AB InBev's own numbers against a named real competitor

**✅ PASS**

In [ ]:
_ = ask("How does AB InBev's overall performance compare to Heineken's public position?", '08')

08
USER: How does AB InBev's overall performance compare to Heineken's public position?

[intent=comparison | sub_agents=['structured', 'web']]
[assumptions/limitations surfaced:]
  - 'Heineken' isn't part of Anheuser-Busch InBev (AB InBev)'s tracked entities (brand/zone/country), or is a different company entirely (e.g. a competitor), so no internal structured data exists for it. Any answer about it, if given, is qualitative/public information only, not internal reporting.

AGENT: ## Short answer

I can't give you a data-backed comparison — the evidence returned for this question is effectively empty, so any numbers I produced would be fabricated.

## What the evidence actually contains

- **Internal KPI data:** the structured query returned *no matching rows* (`fact_kpi` lookup empty).
- **Web search:** no results were returned for this query.
- **Known limitation (as flagged in the evidence):** Heineken is not part of AB InBev's tracked entities (brand / zone / country). It is a sep